# (1) Set up and tokenize Cancer Dependency Map data for embedding extraction

In [3]:
# Imports

import sys
from pathlib import Path
import pandas as pd 
import numpy as np
import scanpy as sc
import anndata as ad

import os
from datasets import load_from_disk

from tqdm import tqdm
import pickle

# change to path to your Geneformer directory
sys.path.append('/work/magroup/kaileyhu/Geneformer')
from geneformer import EmbExtractor
from geneformer import TranscriptomeTokenizer

pd.options.mode.chained_assignment = None 

In [4]:
omics_expr = pd.read_csv("/work/magroup/kaileyhu/Cilantro-SL/data/OmicsExpressionProteinCodingGenesTPMLogp1.csv")
omics_expr.set_index("Unnamed: 0", inplace = True)

metadata = pd.read_csv("/work/magroup/kaileyhu/Cilantro-SL/data/metadata.csv")
metadata.set_index('ModelID', inplace = True)

In [5]:
adata = ad.AnnData(omics_expr)
adata.obs_names = [str(i).split(" ")[0] for i in omics_expr.index]
adata.var_names = [str(i).split(" ")[0] for i in omics_expr.columns]

adata.obs["patient_id"] = omics_expr.index
adata.obs["cell_line"] = metadata["OncotreeLineage"]
adata.obs["disease"] = metadata["OncotreePrimaryDisease"]

for patient in adata.obs_names:
    adata.obs.loc[patient, 'cell_line'] = metadata.loc[patient, "StrippedCellLineName"]
    if (metadata.loc[patient, "OncotreePrimaryDisease"] != "Non-Cancerous"):
        adata.obs.loc[patient, 'disease_state'] = "Cancerous"
    else:
        adata.obs.loc[patient, 'disease_state'] = "Non-Cancerous"
    adata.obs.loc[patient, 'disease'] = metadata.loc[patient, "OncotreePrimaryDisease"]

In [6]:
ensembl_path = "/work/magroup/kaileyhu/Cilantro-SL/data/ensembl_mapping_dict_gc95M.pkl"

def invert_dict(dict_obj):
    return {v: k for k, v in dict_obj.items()}

with open(ensembl_path, "rb") as f:
    id_gene_dict = pickle.load(f)
    gene_id_dict = invert_dict(id_gene_dict)
    
def query_id(g):
    if g in id_gene_dict:
        return id_gene_dict[g]
    else:
        return " "

In [9]:
with open("/work/magroup/kaileyhu/Cilantro-SL/data/Human_SL_pairs.pkl", "rb") as f:
    SL_genes = pickle.load(f)

relevant_genes = SL_genes.keys()

SL_ensembl = [query_id(g) for g in relevant_genes]
SL_ensembl = list(filter(lambda x: x != " ", SL_ensembl))

In [10]:
lst = []
genes = []

for gene in adata.var_names:
    gene2 = gene.split(" ")[0]
    if gene2 in id_gene_dict:
        lst.append(id_gene_dict[gene2])
        genes.append(gene2)
    else:
        lst.append(None)


filtered_results = []

res = []
for val in lst:
    if val is not None:
        filtered_results.append(val)

adata2 = adata[:,genes]
adata2.var_names = filtered_results 
adata2.var['ensembl_id'] = filtered_results
adata2.obs['n_counts'] = adata2.X.sum(axis=1)
adata2.obs["cell_type"] = adata.obs['cell_line']
adata2.obs["disease"] = adata.obs['disease']
adata2.obs["disease_state"] = adata.obs['disease_state']
adata2.obs["batch"] = 0
adata2.obs["patient_id"] = adata.obs["patient_id"]
adata2.var['gene_name'] = genes

In [11]:
adata2.write_h5ad("/work/magroup/kaileyhu/Cilantro-SL/data/processed/omics_expr.h5ad",compression='gzip')

In [12]:
sc.pp.highly_variable_genes(adata2, n_top_genes=500, inplace=True)

In [13]:
hvgs = adata2.var["highly_variable"]

In [14]:
# add SL genes

total_overlap = 0
total_missing = 0
for gene in tqdm(SL_ensembl):
    if gene in hvgs:
        if hvgs[gene]:
            total_overlap += 1
        hvgs[gene] = True
    else:
        total_missing += 1

100%|██████████| 9762/9762 [00:06<00:00, 1598.01it/s]


In [15]:
adata_hvg = adata2[:, hvgs]


save_path = "/work/magroup/kaileyhu/Cilantro-SL/data/processed/SL_hvg/omics_expr_hvg_w_SL.h5ad"
adata_hvg.write_h5ad(save_path)

In [16]:
tk = TranscriptomeTokenizer({"cell_type": "cell_type", "disease": "disease", "disease_state": "disease_state", "patient_id" : "patient_id"}, 
                            nproc=16,
                            special_token = False, 
                            model_input_size=2048)  

<cls> and <eos> are in gene_token_dict but special_token = False. Please note that for 95M model series, special_token should be True.


In [17]:
tk.tokenize_data('/work/magroup/kaileyhu/Cilantro-SL/data/processed/SL_hvg/', 
                 "/work/magroup/kaileyhu/Cilantro-SL/data/processed/", 
                 "hvg_500_w_SL_tokenized_2048", 
                 file_format="h5ad")

Tokenizing /work/magroup/kaileyhu/Cilantro-SL/data/processed/SL_hvg/omics_expr_hvg_w_SL.h5ad
/work/magroup/kaileyhu/Cilantro-SL/data/processed/SL_hvg/omics_expr_hvg_w_SL.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


Map (num_proc=16):   0%|          | 0/1479 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1479 [00:00<?, ? examples/s]